# RedZone world model: speedrun seed check (EXP-109b) + doc-mask retry + FIM-PP (EXP-110), one Colab A100

EXP-109 picked recipe E (Muon + WSD + modded-nanoGPT bundle + torch.compile, 2.07x faster to 0.334) on ONE seed, with
loss gaps of ~0.003. This reruns A_base and E_compile at seeds 1 and 2 (EXP-109 was seed 1234), retries F_docmask
(FlexAttention fixed), and runs EXP-110 with OpenFIM from a pinned commit. Same data, split, 18m preset and 15 min per
arm as EXP-109. Personal use only.

Runtime > Change runtime type > **A100 GPU**, then **Run all**. About 1.6 h (~21 CU): data ~25 min, 5 arms x 15 min,
EXP-110 ~15 min. If the runtime restarts but keeps its disk, Run all again: finished arms are skipped. A failed arm
prints the end of its log. The summary cell prints the seed table and the EXP-109b verdict.


In [ ]:
PIN = "3134544496ca655b9b32b1a185b93b6c698f3f96"   # kaggle-inputs commit: factory code pack (nfm_train.py with the EXP-109 flags)
REPO = "BouncySlime1215/redzone-everything"
LEAGUES = "nfl,nba,mlb"
E = ["--muon", "--sched", "wsd", "--qk-norm", "--relu2", "--untie", "--softcap", "15", "--value-emb", "--compile"]
ARMS = {                       # arm -> (flags, seed); A_s1 runs first (WSD reads its step count)
    "A_s1": ([], 1), "E_s1": (E, 1), "A_s2": ([], 2), "E_s2": (E, 2),
    "F_docmask": (E + ["--doc-mask"], 1234),
}
EXP109 = {"A": {"best": 0.3263, "hours": 0.095}, "E": {"best": 0.3234, "hours": 0.046}}   # seed 1234 (report b8f4fc80)
MINUTES, CTX, BATCH, LR, TARGET = 15, 1024, 64, 6e-4, 0.334     # EXP-103 batch + lr; ctx 1024 (FlexAttention blocks)
TRAIN_TOKENS = 2e9          # never the limit: the 15-min clock is; the WSD decay is set from the steps that fit
RUN, CODE = "/content/rz", "/content/rz/code"
CU_PER_H = {"A100": 13.0, "L4": 5.0, "T4": 2.0}


In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > A100 GPU, then Run all again")
GPU = torch.cuda.get_device_name(0)
RATE = next((v for k, v in CU_PER_H.items() if k in GPU), None)
print("GPU:", GPU, "| CU/h:", RATE)


In [ ]:
import json, os, subprocess, sys, tarfile, time, urllib.request
T_START = time.time()
os.makedirs(CODE, exist_ok=True)
if not os.path.exists(f"{CODE}/rz/scripts/nfm_train.py"):
    for i, ref in enumerate((PIN, PIN[:12], PIN, PIN[:12])):     # the raw CDN can lag a fresh push for a minute
        url = f"https://raw.githubusercontent.com/{REPO}/{ref}/factory/rz_factory_code.tar.gz"
        try:
            print("downloading", url)
            urllib.request.urlretrieve(url, f"{RUN}/code.tar.gz")
            break
        except Exception as e:
            print(" ", type(e).__name__, e)
            time.sleep(30 * i)
    else:
        raise SystemExit("could not download the code pack; Run all again in a minute")
    with tarfile.open(f"{RUN}/code.tar.gz") as t:
        t.extractall(CODE, filter="data")
print("code:", sorted(os.listdir(f"{CODE}/rz")))


In [ ]:
def run(arm, flags, seed, smoke=False):
    out = f"{RUN}/{'smoke' if smoke else 'speedrun'}/{arm}"
    os.makedirs(out, exist_ok=True)
    if os.path.exists(f"{out}/grades.json"):
        print(arm, "done already"); return
    cmd = [sys.executable, "-u", f"{CODE}/rz/scripts/nfm_train.py", "--out", out, "--ckpt-dir", out,
           "--data-dir", f"{RUN}/data", "--build-vocab", "1", "--leagues", LEAGUES, "--preset", "18m",
           "--ctx", str(CTX), "--batch", str(BATCH), "--lr", str(LR), "--warmup", "200",
           "--train-tokens", str(TRAIN_TOKENS), "--minutes", str(MINUTES), "--target-loss", str(TARGET),
           "--throughput-only", "--exp", "EXP-109b", "--seed", str(seed), *flags]
    if smoke:
        cmd += ["--max-steps", "50", "--ckpt-min", "0.2", "--eval-every", "25"]
    if "--sched" in flags and not smoke:      # WSD needs a known end: the steps that fit in 15 min, measured by arm A
        a = f"{RUN}/speedrun/A_s1/grades.json"
        if os.path.exists(a):
            cmd += ["--train-tokens", str(json.load(open(a))["tokens_seen"])]
    with open(f"{out}/train.log", "a") as log:
        p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                             env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in p.stdout:
            print(line, end=""); log.write(line)
        rc = p.wait()
        print(arm, "exit code:", rc)
    if rc != 0 or not os.path.exists(f"{out}/grades.json"):
        print(f"----- {arm} FAILED: last 40 log lines -----")
        print("".join(open(f"{out}/train.log").readlines()[-40:]))
        if smoke:
            print(f"SMOKE FAILED for {arm}: its long run is skipped")

# SMOKE first (~1.5 min per arm): 50 steps + an in-loop checkpoint; an arm that fails it is skipped, not run for 15 min
ok = {}
for arm, (flags, seed) in ARMS.items():
    run(arm, flags, seed, smoke=True)
    ok[arm] = os.path.exists(f"{RUN}/smoke/{arm}/grades.json")
print("smoke:", ok)
for arm, (flags, seed) in ARMS.items():
    if ok[arm]:
        run(arm, flags, seed)


In [ ]:
rows = []
for arm in ARMS:
    gp = f"{RUN}/speedrun/{arm}/grades.json"
    if os.path.exists(gp):
        g = json.load(open(gp))
        t = (g.get("target") or {}).get("reached")
        rows.append((arm, g["best_valid_loss"], g["valid_loss_by_league"], g["tokens_seen"],
                     g["throughput"]["tokens_per_s"], t and t["tokens"], t and t["train_hours"]))
print("arm | best valid | per league | tokens seen | tok/s | tokens to 0.334 | hours to 0.334")
for r in rows:
    print(" | ".join(str(x) for x in r))
json.dump(rows, open(f"{RUN}/speedrun_summary.json", "w"))
print(f"session wall clock: {(time.time() - T_START) / 3600:.2f} h, CU ~{(time.time() - T_START) / 3600 * (RATE or 0):.1f}")

# EXP-109b verdict: E stays if its mean best valid over 3 seeds is below A's AND it reaches 0.334 >= 10% sooner on every seed
G = {arm: json.load(open(f"{RUN}/speedrun/{arm}/grades.json")) for arm in ARMS
     if os.path.exists(f"{RUN}/speedrun/{arm}/grades.json")}
def hrs(g):
    t = (g.get("target") or {}).get("reached")
    return t and t["train_hours"]
same = "A100" in GPU         # EXP-109's seed-1234 numbers are A100 numbers: a 15-min arm on another GPU is not comparable
best = {k: ([EXP109[k]["best"]] if same else []) + [G[f"{k}_s{s}"]["best_valid_loss"] for s in (1, 2) if f"{k}_s{s}" in G]
        for k in "AE"}
hours = {k: ([EXP109[k]["hours"]] if same else []) + [hrs(G[f"{k}_s{s}"]) for s in (1, 2) if f"{k}_s{s}" in G] for k in "AE"}
print("GPU", GPU, "| seed 1234 from EXP-109 included:", same)
print("best valid by seed:", best)
print("hours to 0.334 by seed:", hours)
need = 3 if same else 2
if len(best["A"]) == need and len(best["E"]) == need:
    mean = {k: sum(v) / need for k, v in best.items()}
    sooner = all(a and e and e <= 0.9 * a for a, e in zip(hours["A"], hours["E"]))
    print(f"mean best valid A {mean['A']:.4f} vs E {mean['E']:.4f}; E >= 10% sooner on every seed: {sooner}")
    print("EXP-109b:", "E STAYS the ladder recipe" if mean["E"] < mean["A"] and sooner else "E FAILS: switch the remaining ladder points to H")


In [ ]:
# EXP-110 (~15 min): FIM-PP zero-shot break timing vs the per-mark median, on the same v1 test nights.
# OpenFIM (MIT) from a pinned commit on PYTHONPATH: its pip package pins python < 3.13 and builds with setuptools_scm,
# which is what broke `pip install git+...` in EXP-109's session. Only its runtime imports are installed.
FIM_COMMIT = "cee2bb53be1e8e92b73f98dfdbe3605efc9119cc"
try:
    fim = f"{RUN}/OpenFIM"
    if not os.path.exists(f"{fim}/src/fim"):
        subprocess.run(["git", "clone", "-q", "https://github.com/FIM4Science/OpenFIM", fim], check=True)
    subprocess.run(["git", "-C", fim, "checkout", "-q", FIM_COMMIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.56.2", "huggingface_hub==0.36.2",
                    "accelerate", "einops", "torchdiffeq", "torchsde", "optree", "pydantic", "h5py", "python-dotenv",
                    "loguru", "torchinfo", "peft", "datasets", "psutil", "gitpython", "scikit-learn", "pyyaml"], check=True)
    env = {**os.environ, "PYTHONPATH": f"{fim}/src" + os.pathsep + os.environ.get("PYTHONPATH", "")}
    for lg in ("nba", "nfl"):          # NBA first; MLB (Retrosheet) has no clock to time breaks with
        out = f"{RUN}/exp110/{lg}"
        p = subprocess.run([sys.executable, "-u", f"{CODE}/rz/scripts/exp110_fimpp.py", "--tokens", f"{RUN}/data/tokens/{lg}",
                            "--leagues", lg, "--out", out, "--max-test", "100"], cwd=RUN, text=True, env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)   # Colab hides a child's output: show it
        os.makedirs(out, exist_ok=True)
        open(f"{out}/exp110.log", "w").write(p.stdout)
        print("".join(p.stdout.splitlines(True)[-60:]))
        if os.path.exists(f"{out}/grades.json"):
            print(open(f"{out}/grades.json").read())
        else:
            print(lg, "EXP-110 gave no grades.json (exit", p.returncode, ")")
except Exception as e:
    print("EXP-110 did not run:", type(e).__name__, e, "(the speedrun results above are unaffected)")
